# 05 · Probabilidad condicional e independencia: ejercicios resueltos

Tres ejercicios sobre dos dados y una urna. Cada afirmación se demuestra o se refuta con un contraejemplo explícito, que el código verifica.

In [1]:
from fractions import Fraction
from itertools import product

Omega = set(product(range(1, 7), repeat=2))


def P(A, Omega=Omega):
    return Fraction(len(A & Omega), len(Omega))


def P_cond(A, B):
    return P(A & B) / P(B)


def independientes(A, B):
    return P(A & B) == P(A) * P(B)

## Ejercicio 1

Se lanzan dos dados honestos. Para $n\in R_S=\{2,\dots,12\}$ y $m\in R_D=\{0,\dots,5\}$ sean

- $S_n$: la suma de las caras es $n$,
- $D_m$: la diferencia (en valor absoluto) de las caras es mayor o igual a $m$.

1. Calcula $\mathbb{P}(S_7\mid D_3)$ y $\mathbb{P}(D_3\mid S_7)$.
2. ¿Para qué $(n,m)$ son ajenos $S_n$ y $D_m$?
3. ¿Para qué $(n,m)$ son independientes?
4. Calcula $\mathbb{P}(S_n\cap D_m)$ para todos los pares.

**Solución.** El espacio muestral es $\Omega=\{1,\dots,6\}^2$ y

$$S_n=\{(i,j)\in\Omega: i+j=n\},\qquad D_m=\{(i,j)\in\Omega: |i-j|\ge m\}.$$

In [2]:
def S(n):
    return {w for w in Omega if w[0] + w[1] == n}

def D(m):
    return {w for w in Omega if abs(w[0] - w[1]) >= m}

R_S, R_D = range(2, 13), range(0, 6)
print("P(S7 | D3) =", P_cond(S(7), D(3)))
print("P(D3 | S7) =", P_cond(D(3), S(7)))

P(S7 | D3) = 1/3
P(D3 | S7) = 2/3


$S_7=\{(1,6),(2,5),(3,4),(4,3),(5,2),(6,1)\}$ y de estas parejas cuatro tienen diferencia al menos $3$; $\#D_3=12$. Así $\mathbb{P}(S_7\mid D_3)=4/12=1/3$ y $\mathbb{P}(D_3\mid S_7)=4/6=2/3$.

**Eventos ajenos.** $i+j$ e $|i-j|$ tienen la misma paridad, y $|i-j|\le\min(i+j-2,\,12-i-j)$. Los pares ajenos son:

In [3]:
ajenos = sorted((n, m) for n in R_S for m in R_D if not S(n) & D(m))
print(len(ajenos), "pares ajenos:", ajenos)

30 pares ajenos: [(2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (3, 2), (3, 3), (3, 4), (3, 5), (4, 3), (4, 4), (4, 5), (5, 4), (5, 5), (6, 5), (8, 5), (9, 4), (9, 5), (10, 3), (10, 4), (10, 5), (11, 2), (11, 3), (11, 4), (11, 5), (12, 1), (12, 2), (12, 3), (12, 4), (12, 5)]


**Independencia.** $D_0=\Omega$, así que $S_n$ y $D_0$ son independientes para todo $n$. El código muestra que no hay otros pares:

In [4]:
sorted((n, m) for n in R_S for m in R_D if independientes(S(n), D(m)))

[(2, 0),
 (3, 0),
 (4, 0),
 (5, 0),
 (6, 0),
 (7, 0),
 (8, 0),
 (9, 0),
 (10, 0),
 (11, 0),
 (12, 0)]

In [5]:
import pandas as pd

tabla = pd.DataFrame({m: {n: P(S(n) & D(m)) for n in R_S} for m in R_D})
tabla.index.name, tabla.columns.name = "n", "m"
tabla

m,0,1,2,3,4,5
n,,,,,,
2,1/36,0,0,0,0,0
3,1/18,1/18,0,0,0,0
4,1/12,1/18,1/18,0,0,0
5,1/9,1/9,1/18,1/18,0,0
6,5/36,1/9,1/9,1/18,1/18,0
7,1/6,1/6,1/9,1/9,1/18,1/18
8,5/36,1/9,1/9,1/18,1/18,0
9,1/9,1/9,1/18,1/18,0,0
10,1/12,1/18,1/18,0,0,0


## Ejercicio 2

Sean $A$, $B$ y $C$ eventos con probabilidades positivas. ¿Cuáles afirmaciones son verdaderas?

a) $\mathbb{P}(A\mid B)+\mathbb{P}(A^c\mid B)=1$.

b) $\mathbb{P}(A\mid B)+\mathbb{P}(A\mid B^c)=1$.

c) Si $A$ y $B$ son independientes, entonces $\mathbb{P}(A\cap B\mid C)=\mathbb{P}(A\mid C)\,\mathbb{P}(B\mid C)$.

d) Si $\mathbb{P}(A\mid B)=\mathbb{P}(B)$, entonces $A$ y $B$ son independientes.

e) Si $\mathbb{P}(A)=\mathbb{P}(B)=p$, entonces $\mathbb{P}(A\cap B)\le p^2$.

**a) Verdadera.** $\mathbb{P}(\,\cdot\mid B)$ es una medida de probabilidad, y $A$, $A^c$ forman una partición de $\Omega$.

**b) Falsa.** Con $A=S_7$ y $B=D_2$:

In [6]:
Dc2 = Omega - D(2)
P_cond(S(7), D(2)), P_cond(S(7), Dc2), P_cond(S(7), D(2)) + P_cond(S(7), Dc2)

(Fraction(1, 5), Fraction(1, 8), Fraction(13, 40))

**c) Falsa.** La independencia no se conserva al condicionar. Sean $A$: «el primer dado es par», $B$: «el segundo dado es par» y $C$: «la suma es par». $A$ y $B$ son independientes, pero dado $C$ los dos dados tienen la misma paridad:

$$\mathbb{P}(A\cap B\mid C)=\tfrac12\neq\tfrac14=\mathbb{P}(A\mid C)\,\mathbb{P}(B\mid C).$$

In [7]:
A = {w for w in Omega if w[0] % 2 == 0}
B = {w for w in Omega if w[1] % 2 == 0}
C = {w for w in Omega if sum(w) % 2 == 0}
print(independientes(A, B), P_cond(A & B, C), P_cond(A, C) * P_cond(B, C))

True 1/2 1/4


**d) Falsa.** Sean $B$: «el primer dado es a lo más $3$» y $A$: «los dos dados son a lo más $3$». Entonces $A\subseteq B$, $\mathbb{P}(A\mid B)=\frac12=\mathbb{P}(B)$, pero $\mathbb{P}(A\cap B)=\frac14\neq\frac18=\mathbb{P}(A)\mathbb{P}(B)$.

In [8]:
B = {w for w in Omega if w[0] <= 3}
A = {w for w in Omega if max(w) <= 3}
print(P_cond(A, B) == P(B), independientes(A, B))

True False


**e) Falsa.** $\mathbb{P}(S_7)=\mathbb{P}(D_4)=\frac16$, pero $S_7\cap D_4=\{(1,6),(6,1)\}$ tiene probabilidad $\frac1{18}>\frac1{36}$. Lo que sí vale siempre es $\mathbb{P}(A\cap B)\le p$.

In [9]:
P(S(7)), P(D(4)), P(S(7) & D(4)), P(S(7) & D(4)) <= P(D(4)) ** 2

(Fraction(1, 6), Fraction(1, 6), Fraction(1, 18), False)

## Ejercicio 3

Una urna contiene $10$ bolas numeradas del $1$ al $10$. Se extraen dos bolas al azar, sin reemplazo y consecutivamente. Calcula

a) la probabilidad de que la suma de los dos números sea par;

b) la probabilidad de que la primera bola sea par dado que la suma fue par.

**Solución.** El espacio muestral son las parejas ordenadas de números distintos, $\#\Omega=10\cdot 9=90$. Sean $B$: «la suma es par» y $A_1$: «la primera bola es par». La suma es par si y solo si ambas bolas tienen la misma paridad:

$$\mathbb{P}(B)=\mathbb{P}(A_1)\mathbb{P}(B\mid A_1)+\mathbb{P}(A_1^c)\mathbb{P}(B\mid A_1^c)=\tfrac12\cdot\tfrac49+\tfrac12\cdot\tfrac49=\tfrac49.$$

Por la fórmula de Bayes,

$$\mathbb{P}(A_1\mid B)=\frac{\mathbb{P}(B\mid A_1)\mathbb{P}(A_1)}{\mathbb{P}(B)}=\frac{2/9}{4/9}=\frac12.$$

In [10]:
Omega_urna = {(i, j) for i in range(1, 11) for j in range(1, 11) if i != j}
B = {w for w in Omega_urna if sum(w) % 2 == 0}
A1 = {w for w in Omega_urna if w[0] % 2 == 0}
P(B, Omega_urna), Fraction(len(A1 & B), len(B))

(Fraction(4, 9), Fraction(1, 2))